# 🧩 Tokenizer Exploration

Comprehensive evaluation and optimization of tokenization strategies for MRI text data processing.

---

## 🎯 Objectives

- 🚀 **Strategy Comparison:** Systematic evaluation of popular pretrained tokenizers and preprocessing approaches
- 📝 **Domain Adaptation:** Custom tokenizer training on the complete MRI text dataset for optimal performance
- 🏆 **Performance Optimization:** Comparative analysis to identify the most effective tokenization method for this specific domain

## 📋 Implementation

- **🔬 Tokenizer Evaluation:** Comprehensive testing of three major tokenization approaches:
  - **BPE (Byte-Pair Encoding):** Standard subword tokenization method
  - **WordPiece:** Google's tokenization algorithm used in BERT
  - **Unigram:** Probabilistic subword tokenization approach
- **📊 Training Data:** `train_text.txt` contains the complete MRI text corpus including:
  - BodyRegion_exam, Program_exam, LeanExamination_exam
  - BodyRegion_meas, Protocol_meas, LeanProtocol_meas, Coils_meas
- **🛠️ Preprocessing Pipeline:** Advanced text normalization and special token handling:
  - NaN values were replaced with a special `[NAN]` token to enable proper <br>
    tokenization and consistent sequence lengths.
  - Special characters were normalized before tokenization: <br>
    `,` or `;` with [SEP], `_`, `%\`, `%/`, `%` with [SUBSEQ], `/` with [SUB], and <br>
    `+` with [AND]
  - Numbered bracketed patterns like `[/12]` or `[3]` were converted to <br>
    `[REPEAT]12` or `[REPEAT]3` so repeated protocol fragments can be <br>
    recognized consistently.
  - Text normalization with lowercase conversion and accent removal
  - Strategic pattern splitting for MRI-specific terminology


Replace the  in the text columns to create special tokens that can be used to

## 🏆 Results

**WordPiece Tokenizer** emerged as the optimal solution for this MRI domain:
- Superior handling of medical terminology and technical sequences
- Effective subword segmentation for MRI-specific compound terms
- Best balance between vocabulary size and semantic preservation
- Robust performance on domain-specific special tokens and patterns

The trained WordPiece tokenizer is saved as `tokenizer_wordpiece.json` and <br>
integrated into the multimodal dataset pipeline.

In [ ]:
import os
import sys

# Get the current working directory
CWD = os.getcwd()

# Define the path to the local utils directory
LOCA_UTILS_DIR = os.path.normpath(os.path.join(CWD, "..", "Tokenization utils_opt"))
# Define the path to the global utils directory
GLOBAL_UTILS_DIR = os.path.normpath(
    os.path.join(CWD, "..", "..", "..", "..", "Global utils_opt")
)

# Append this directory to the system path
sys.path.append(LOCA_UTILS_DIR)
sys.path.append(GLOBAL_UTILS_DIR)


from global_utils import *
from tokenization_utils import *

# Enable autoreload and black formatting in Jupyter notebooks
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

✅ Found style_sheet_matplotlib.mplstyle at: /home/raraabf1/PhD/MRI_data_analysis/Stylesheets/style_sheet_matplotlib.mplstyle
✅ Found style_sheet_plotly.json at: /home/raraabf1/PhD/MRI_data_analysis/Stylesheets/style_sheet_plotly.json


In [2]:
# Use a pre-trained tokenizer from Hugging Face
from tokenizers import (
    decoders,
    models,
    pre_tokenizers,
    trainers,
    Tokenizer,
    normalizers,
)

## 📦 Data Loading and Preprocessing
- 🗃️ Load the preprocessed data from the pickle file
- 🧹 Filter the dataset to remove unwanted entries
- 🏷️ Rename scanner identifiers for clarity
- 📊 Extract relevant parameters from the dataframes

In [ ]:
# Define the data directory path
DATA_DIR = os.path.normpath(
    os.path.join(
        CWD,
        "..",
        "..",
        "..",
        "..",
        "saved_datasets",
        "preprocessing",
    )
)

# Read in the raw MRI data from both pickle and parquet file
RAW_MRI_PKL_DF = data_read_in(DATA_DIR, "processed_mri_data.pkl")
RAW_MRI_PARQ_DF = data_read_in(DATA_DIR, "processed_mri_data.parquet")

# Create and rename necessary columns as ScanDuration_s_meas and ScanDuration_s_exam
RAW_MRI_PKL_DF = create_rename_columns(RAW_MRI_PKL_DF)
RAW_MRI_PARQ_DF = create_rename_columns(RAW_MRI_PARQ_DF)

# Perform filtering steps and plot the funnel chart for the filtering steps
# Filter out 0s and NaNs, exclude long examinations/measurements
FILTERED_MRI_PKL_DF, FILTERING_DICT = filter_dataframes(RAW_MRI_PKL_DF)
FILTERED_MRI_PARQ_DF, _ = filter_dataframes(RAW_MRI_PARQ_DF)


# Replace the Serial_scan values with the scanner names first
SCANNER_MAPPING = {
    69667: "AvantoFit (CRONA)",
    183811: "Sola (CRONA)",
    142185: "Aera (CRONA)",
    202017: "VidaFit (CRONA)",
    167008: "PrismaFit (CRONA)",
    75609: "Vida (CRONA)",
    142082: "Aera (UFK)",
}

# Apply the scanner mapping to both DataFrames using the Serial_scan column
MRI_PKL_DF = apply_scanner_mapping(
    df=FILTERED_MRI_PKL_DF,
    scanner_mapping=SCANNER_MAPPING,
    column="Serial_scan",
)
MRI_PARQ_DF = apply_scanner_mapping(
    df=FILTERED_MRI_PARQ_DF,
    scanner_mapping=SCANNER_MAPPING,
    column="Serial_scan",
)

# Apply the body region grouping to both DataFrames using the BodyRegion_meas column
MRI_PKL_DF = apply_bodyregion_grouping(MRI_PKL_DF, BODYREGION_GROUPING)
MRI_PARQ_DF = apply_bodyregion_grouping(MRI_PARQ_DF, BODYREGION_GROUPING)

# Sort the columns in both DataFrames first by suffix priority and then alphabetically
SUFFIX_PRIOTIRY = {"_scan": 0, "_exam": 1, "_meas": 2, "_param": 3, "_energy": 4}
MRI_PKL_DF = reorder_columns_by_suffix_priority(MRI_PKL_DF, SUFFIX_PRIOTIRY)
MRI_PARQ_DF = reorder_columns_by_suffix_priority(MRI_PARQ_DF, SUFFIX_PRIOTIRY)

1. Initial number of rows: 600293
2. Dropped 0 rows with NaN values.
3. Dropped 0 rows with 0 values.
4. Dropped 359 rows due to examination duration.
5. Dropped 402 rows due to measurement duration.
🧹 Filtered 761 rows from the DataFrame.
✅ Number of rows after filtering: 599532
1. Initial number of rows: 600293
2. Dropped 0 rows with NaN values.
3. Dropped 0 rows with 0 values.
4. Dropped 359 rows due to examination duration.
5. Dropped 402 rows due to measurement duration.
🧹 Filtered 761 rows from the DataFrame.
✅ Number of rows after filtering: 599532
⚠️ The following body regions are not mapped in BODYREGION_GROUPING and will be categorized as 'Other': {None}
⚠️ The following body regions are not mapped in BODYREGION_GROUPING and will be categorized as 'Other': {None}


## 📝 Text Features
- Train a custom tokenizer on the entire text dataset.

## 🧩 Tokenizer Strategy
- 🚀 Use a **pretrained tokenizer** as a starting point.
- 📚 Train the tokenizer on the complete text data for domain adaptation.
- 🏆 Prefer the **WordPiece Tokenizer** (best performance in this domain).
- 📏 In your custom dataset, ensure the **token sequence length for each feature is constant**.
- 🧱 Pad sequences with a special **[PAD]** token and always end each feature <br>
   with a **tab (`\t`)** token, so the encoder knows where each feature ends.

In [ ]:
TEXT_COLUMNS = [
    "Machine_scan",
    "AllBodyRegions_exam",
    "BodyRegion_exam",
    "Program_exam",
    "UsedAddins_exam",
    "Addin_meas",
    "AdjustmentType_meas",
    "BodyRegionGroup_meas",
    "BodyRegion_meas",
    "Coils_meas",
    "LeanProtocol_meas",
    "Protocol_meas",
    "Sequence_meas",
]

TEXT_DF = MRI_PKL_DF[TEXT_COLUMNS]
display(TEXT_DF)

,Machine_scan,AllBodyRegions_exam,BodyRegion_exam,Program_exam,UsedAddins_exam,Addin_meas,AdjustmentType_meas,BodyRegionGroup_meas,BodyRegion_meas,Coils_meas,LeanProtocol_meas,Protocol_meas,Sequence_meas
0,AvantoFit (CRONA),HEAD,HEAD,Neuro_Rad/Kopf_NR/Routine/Metastasen-Kurzprogramm,None,AutoAlign,None,Neuroradiology,HEAD,HE1-4,t2_flair_tra,t2_flair_tra_4mm,%SiemensSeq%\tse
1,AvantoFit (CRONA),HEAD,HEAD,Neuro_Rad/Kopf_NR/Routine/Metastasen-Kurzprogramm,None,AutoAlign,None,Neuroradiology,HEAD,HE1-4,t1_se_tra,t1_se_tra_4mm,%SiemensSeq%\se
...,...,...,...,...,...,...,...,...,...,...,...,...,...
599530,VidaFit (CRONA),BRAIN,BRAIN,6_VidaFit_Forschung/Forschung_Neurorad/Buschi/...,None,AutoAlign,AdjFre,Neuroradiology,BRAIN,"HEA,HEP BC",AdjFre,AdjFre,%AdjustSeq%/AdjFreSeq
599531,VidaFit (CRONA),BRAIN,BRAIN,6_VidaFit_Forschung/Forschung_Neurorad/Buschi/...,None,None,None,Neuroradiology,BRAIN,"HEA,HEP",diff_ep_2d_fs,hc_b-1400_ep2d_diff_32av_adv shim-B1 volselect...,%SiemensSeq%\ep2d_diff


In [5]:
# Replace the NaN values in the text columns with [NAN] to avoid issues during tokenization
TEXT_DF = TEXT_DF.fillna("[NAN]")

# Replace the , or ; with [SEP], _, %\, %/, % with [SUBSEQ], / with [SUB], and
# + with [AND] in the text columns to create special tokens that can be used to
# separate different parts of the text during tokenization. The regex=True argument
TEXT_DF = TEXT_DF.replace(
    {r",|;": "[SEP]", r"_|%\\|%/|%": "[SUBSEQ]", "/": "[SUB]", r"\+": "[AND]"},
    regex=True,
)

# Replace the "[/Number]" with "[REPEAT] /Number", \d+ captures one or more digits
# inside the square brackets and appendes it to the string "[REPEAT] "
TEXT_DF = TEXT_DF.replace(r"\[(\d+)\]", r"[REPEAT]\1", regex=True)


# Remove any [SUBSEQ] or [SUB] at the beginning of a string
TEXT_DF = TEXT_DF.replace(r"^\[SUBSEQ\]|^\[SUB\]", "", regex=True)

# Remove duplicate following [SEP], [SUBSEQ], [SUB], and [AND] tokens
TEXT_DF = TEXT_DF.replace(
    r"(\[(?:SEP|SUBSEQ|SUB|AND)\])(?:\1)+",
    r"\1",
    regex=True,
)

display(TEXT_DF)

,Machine_scan,AllBodyRegions_exam,BodyRegion_exam,Program_exam,UsedAddins_exam,Addin_meas,AdjustmentType_meas,BodyRegionGroup_meas,BodyRegion_meas,Coils_meas,LeanProtocol_meas,Protocol_meas,Sequence_meas
0,AvantoFit (CRONA),HEAD,HEAD,Neuro[SUBSEQ]Rad[SUB]Kopf[SUBSEQ]NR[SUB]Routin...,[NAN],AutoAlign,[NAN],Neuroradiology,HEAD,HE1-4,t2[SUBSEQ]flair[SUBSEQ]tra,t2[SUBSEQ]flair[SUBSEQ]tra[SUBSEQ]4mm,SiemensSeq[SUBSEQ]tse
1,AvantoFit (CRONA),HEAD,HEAD,Neuro[SUBSEQ]Rad[SUB]Kopf[SUBSEQ]NR[SUB]Routin...,[NAN],AutoAlign,[NAN],Neuroradiology,HEAD,HE1-4,t1[SUBSEQ]se[SUBSEQ]tra,t1[SUBSEQ]se[SUBSEQ]tra[SUBSEQ]4mm,SiemensSeq[SUBSEQ]se
...,...,...,...,...,...,...,...,...,...,...,...,...,...
599530,VidaFit (CRONA),BRAIN,BRAIN,6[SUBSEQ]VidaFit[SUBSEQ]Forschung[SUB]Forschun...,[NAN],AutoAlign,AdjFre,Neuroradiology,BRAIN,HEA[SEP]HEP BC,AdjFre,AdjFre,AdjustSeq[SUBSEQ]AdjFreSeq
599531,VidaFit (CRONA),BRAIN,BRAIN,6[SUBSEQ]VidaFit[SUBSEQ]Forschung[SUB]Forschun...,[NAN],[NAN],[NAN],Neuroradiology,BRAIN,HEA[SEP]HEP,diff[SUBSEQ]ep[SUBSEQ]2d[SUBSEQ]fs,hc[SUBSEQ]b-1400[SUBSEQ]ep2d[SUBSEQ]diff[SUBSE...,SiemensSeq[SUBSEQ]ep2d[SUBSEQ]diff


In [6]:
# Join multiple text columns into a single string for each row
TEXT_LINES = TEXT_DF[TEXT_COLUMNS].astype(str).agg("\t".join, axis=1)

# Save to a txt file, one line per row
TEXT_LINES.to_csv("train_text.txt", index=False, header=False)

In [7]:
# Initialize a tokenizer with a Byte-Pair Encoding (BPE) model
TOKENIZER = Tokenizer(models.BPE())

# Use a normalizer to cast them to lowercase, strip whitespace, and remove accents
TOKENIZER.normalizer = normalizers.Sequence(
    [
        normalizers.NFD(),
        normalizers.Lowercase(),
        normalizers.StripAccents(),
        normalizers.Strip(),
    ]
)


# Define your special tokens
SPECIAL_TOKENS = [
    "[PAD]",
    "[NAN]",
    "[SEP]",
    "[SUBSEQ]",
    "[SUB]",
    "[REPEAT]",
    "[UNK]",
    "[AND]",
]


# Split the text at the special tokens, whitespace, and tabs to create the
# pre-tokenizer
TOKENIZER.pre_tokenizer = pre_tokenizers.Sequence(
    [
        pre_tokenizers.Split(pattern="[SUBSEQ]", behavior="isolated"),
        pre_tokenizers.Split(pattern="[SUB]", behavior="isolated"),
        pre_tokenizers.Split(pattern="[SEP]", behavior="isolated"),
        pre_tokenizers.Whitespace(),
        pre_tokenizers.Split(pattern=r"[ \t]", behavior="isolated"),
    ]
)


# Train the tokenizer on the text file
TRAINER = trainers.BpeTrainer(
    special_tokens=SPECIAL_TOKENS,
    vocab_size=10000,
    min_frequency=2,
    limit_alphabet=1000,
)

# Train the tokenizer on the text file
TOKENIZER.train(files=["train_text.txt"], trainer=TRAINER)

# Optional: set the decoder for ByteLevel tokenization
TOKENIZER.decoder = decoders.ByteLevel()

In [13]:
# Test encoding
SAMPLE_TEXT = "AvantoFit (CRONA)	HEAD	HEAD	Neuro[SUBSEQ]Rad[SUB]Kopf[SUBSEQ]NR[SUB]Routine[SUB]Metastasen-Kurzprogramm	[NAN]	AutoAlign 	[NAN]	Neuroradiology	HEAD	 HE1-4	diff	diff[SUBSEQ]0[SUBSEQ]500[SUBSEQ]1000[SUBSEQ]ADC[SUBSEQ]4scan[SUBSEQ]trace	SiemensSeq[SUBSEQ]ep2d[SUBSEQ]diff"
ECODED = TOKENIZER.encode(SAMPLE_TEXT)
print("Encoded IDs:", ECODED.ids)
print("Encoded tokens:", ECODED.tokens)
# Test decoding
DECODED = TOKENIZER.decode(ECODED.ids, skip_special_tokens=False)
print("Decoded text:", DECODED)

Encoded IDs: [66, 25, 20, 18, 28, 22, 32, 27, 30, 30, 72, 3, 111, 4, 77, 25, 3, 12, 21, 4, 21, 121, 20, 12, 9, 4, 296, 16, 8, 18, 16, 8, 9, 12, 26, 123, 21, 211, 1, 50, 1, 44, 20, 19, 60, 51, 30, 36, 9, 23, 26, 40, 84, 84, 3, 113, 3, 201, 3, 168, 3, 16, 43, 22, 3, 172, 3, 75, 22, 9, 49, 9, 15, 3, 9, 17, 29, 43, 3, 84]
Encoded tokens: ['avanto', 'f', 'i', 't', '(', 'c', 'rona', ')', 'head', 'head', 'neuro', '[SUBSEQ]', 'rad', '[SUB]', 'kop', 'f', '[SUBSEQ]', 'n', 'r', '[SUB]', 'r', 'out', 'i', 'n', 'e', '[SUB]', 'met', 'a', 's', 't', 'a', 's', 'e', 'n', '-', 'kurzprog', 'r', 'amm', '[NAN]', 'autoalign', '[NAN]', 'neurorad', 'i', 'o', 'log', 'y', 'head', 'h', 'e', '1', '-', '4', 'diff', 'diff', '[SUBSEQ]', '0', '[SUBSEQ]', '500', '[SUBSEQ]', '1000', '[SUBSEQ]', 'a', 'd', 'c', '[SUBSEQ]', '4scan', '[SUBSEQ]', 'tra', 'c', 'e', 'siemenss', 'e', 'q', '[SUBSEQ]', 'e', 'p', '2', 'd', '[SUBSEQ]', 'diff']
Decoded text: avanto f i t ( c rona ) head head neuro [SUBSEQ] rad [SUB] kop f [SUBSEQ] n r

In [14]:
# Use WordPiece model instead of BPE
TOKENIZER = Tokenizer(models.WordPiece())


# Use a normalizer to cast them to lowercase, strip whitespace, and remove accents
TOKENIZER.normalizer = normalizers.Sequence(
    [
        normalizers.NFD(),
        normalizers.Lowercase(),
        normalizers.StripAccents(),
        normalizers.Strip(),
    ]
)


# Define your special tokens
SPECIAL_TOKENS = [
    "[PAD]",
    "[NAN]",
    "[SEP]",
    "[SUBSEQ]",
    "[SUB]",
    "[REPEAT]",
    "[UNK]",
    "[AND]",
]


# Split the text at the special tokens, whitespace, and tabs to create the
# pre-tokenizer
TOKENIZER.pre_tokenizer = pre_tokenizers.Sequence(
    [
        pre_tokenizers.Split(pattern="[SUBSEQ]", behavior="isolated"),
        pre_tokenizers.Split(pattern="[SUB]", behavior="isolated"),
        pre_tokenizers.Split(pattern="[SEP]", behavior="isolated"),
        pre_tokenizers.Whitespace(),
        pre_tokenizers.Split(pattern=r"[ \t]", behavior="isolated"),
    ]
)

# Train the tokenizer on the text file
TRAINER = trainers.WordPieceTrainer(
    special_tokens=SPECIAL_TOKENS,
    vocab_size=10000,
    min_frequency=2,
    limit_alphabet=1000,
)

# Train the tokenizer on the text file
TOKENIZER.train(files=["train_text.txt"], trainer=TRAINER)

# Optional: set the decoder for WordPiece
TOKENIZER.decoder = decoders.WordPiece()

# Save the tokenizer to a file
TOKENIZER.save("tokenizer_wordpiece.json")

In [15]:
# Test encoding
ENCODED = TOKENIZER.encode(SAMPLE_TEXT)
print("Encoded IDs:", ENCODED.ids)
print("Encoded tokens:", ENCODED.tokens)
# Test decoding
DECODED = TOKENIZER.decode(ENCODED.ids, skip_special_tokens=False)
print("Decoded text:", DECODED)

Encoded IDs: [269, 12, 135, 13, 137, 137, 166, 3, 291, 4, 253, 3, 277, 4, 256, 4, 444, 15, 760, 1, 262, 1, 213, 137, 225, 15, 21, 325, 325, 3, 17, 3, 713, 3, 642, 3, 701, 3, 651, 3, 666, 185, 3, 397, 3, 325]
Encoded tokens: ['avantofit', '(', 'crona', ')', 'head', 'head', 'neuro', '[SUBSEQ]', 'rad', '[SUB]', 'kopf', '[SUBSEQ]', 'nr', '[SUB]', 'routine', '[SUB]', 'metastasen', '-', 'kurzprogramm', '[NAN]', 'autoalign', '[NAN]', 'neuroradiology', 'head', 'he1', '-', '4', 'diff', 'diff', '[SUBSEQ]', '0', '[SUBSEQ]', '500', '[SUBSEQ]', '1000', '[SUBSEQ]', 'adc', '[SUBSEQ]', '4scan', '[SUBSEQ]', 'trace', 'siemensseq', '[SUBSEQ]', 'ep2d', '[SUBSEQ]', 'diff']
Decoded text: avantofit ( crona ) head head neuro [SUBSEQ] rad [SUB] kopf [SUBSEQ] nr [SUB] routine [SUB] metastasen - kurzprogramm [NAN] autoalign [NAN] neuroradiology head he1 - 4 diff diff [SUBSEQ] 0 [SUBSEQ] 500 [SUBSEQ] 1000 [SUBSEQ] adc [SUBSEQ] 4scan [SUBSEQ] trace siemensseq [SUBSEQ] ep2d [SUBSEQ] diff


In [16]:
# Initialize a Unigram tokenizer
TOKENIZER = Tokenizer(models.Unigram())

# Use a normalizer to cast them to lowercase, strip whitespace, and remove accents
TOKENIZER.normalizer = normalizers.Sequence(
    [
        normalizers.NFD(),
        normalizers.Lowercase(),
        normalizers.StripAccents(),
        normalizers.Strip(),
    ]
)


# Define your special tokens
SPECIAL_TOKENS = [
    "[PAD]",
    "[NAN]",
    "[SEP]",
    "[SUBSEQ]",
    "[SUB]",
    "[REPEAT]",
    "[UNK]",
    "[AND]",
]


# Split the text at the special tokens, whitespace, and tabs to create the
# pre-tokenizer
TOKENIZER.pre_tokenizer = pre_tokenizers.Sequence(
    [
        pre_tokenizers.Split(pattern="[SUBSEQ]", behavior="isolated"),
        pre_tokenizers.Split(pattern="[SUB]", behavior="isolated"),
        pre_tokenizers.Split(pattern="[SEP]", behavior="isolated"),
        pre_tokenizers.Whitespace(),
        pre_tokenizers.Split(pattern=r"[ \t]", behavior="isolated"),
    ]
)

# Set up the trainer for Unigram
TRAINER = trainers.UnigramTrainer(
    special_tokens=SPECIAL_TOKENS,
    vocab_size=10000,
)

# Train the tokenizer on your data
TOKENIZER.train(files=["train_text.txt"], trainer=TRAINER)

In [17]:
# Test encoding
ENCODED = TOKENIZER.encode(SAMPLE_TEXT)
print("Encoded IDs:", ENCODED.ids)
print("Encoded tokens:", ENCODED.tokens)
DECODED = TOKENIZER.decode(ENCODED.ids, skip_special_tokens=False)
print("Decoded text:", DECODED)

Encoded IDs: [66, 25, 20, 18, 28, 22, 32, 27, 30, 30, 72, 3, 111, 4, 77, 25, 3, 12, 21, 4, 21, 121, 20, 12, 9, 4, 296, 16, 8, 18, 16, 8, 9, 12, 26, 123, 21, 211, 1, 50, 1, 44, 20, 19, 60, 51, 30, 36, 9, 23, 26, 40, 84, 84, 3, 113, 3, 201, 3, 168, 3, 16, 43, 22, 3, 172, 3, 75, 22, 9, 49, 9, 15, 3, 9, 17, 29, 43, 3, 84]
Encoded tokens: ['avanto', 'f', 'i', 't', '(', 'c', 'rona', ')', 'head', 'head', 'neuro', '[SUBSEQ]', 'rad', '[SUB]', 'kop', 'f', '[SUBSEQ]', 'n', 'r', '[SUB]', 'r', 'out', 'i', 'n', 'e', '[SUB]', 'met', 'a', 's', 't', 'a', 's', 'e', 'n', '-', 'kurzprog', 'r', 'amm', '[NAN]', 'autoalign', '[NAN]', 'neurorad', 'i', 'o', 'log', 'y', 'head', 'h', 'e', '1', '-', '4', 'diff', 'diff', '[SUBSEQ]', '0', '[SUBSEQ]', '500', '[SUBSEQ]', '1000', '[SUBSEQ]', 'a', 'd', 'c', '[SUBSEQ]', '4scan', '[SUBSEQ]', 'tra', 'c', 'e', 'siemenss', 'e', 'q', '[SUBSEQ]', 'e', 'p', '2', 'd', '[SUBSEQ]', 'diff']
Decoded text: avanto f i t ( c rona ) head head neuro [SUBSEQ] rad [SUB] kop f [SUBSEQ] n r